In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import koreanize_matplotlib
import os
import glob
import pandas as pd
import datetime
import numpy as np
import polars as pl

In [ ]:
import gc


def get_parquet_file_sizes(dir_path):
    file_sizes = {}
    for filename in os.listdir(dir_path):
        if filename.endswith('.parquet'):
            file_path = os.path.join(dir_path, filename)
            file_size_bytes = os.path.getsize(file_path)
            file_size_mb = file_size_bytes / (1024 * 1024)
            file_sizes[filename] = file_size_mb
    return file_sizes


DATA_DIR = Path(r"C:/Users/user/Desktop/playground/codeit_DA_13th/projects/intermediate1/midproject1/DBsources/topic2")
PARQUET_FILES = {path.name.lower(): path for path in DATA_DIR.glob('*.parquet')}


def resolve_data_path(file_name):
    parquet_name = file_name.replace('.csv', '.parquet').lower()
    return str(PARQUET_FILES[parquet_name])


def read_topic2_parquet(file_name, **kwargs):
    return pd.read_parquet(resolve_data_path(file_name), dtype_backend='pyarrow', **kwargs)


def object_top_frequency(df):
    obj_cols = df.select_dtypes(include=['object', 'string']).columns
    if len(obj_cols) == 0:
        return pd.Series(dtype='object')
    return df[obj_cols].describe().loc['top']


def plot_missing_matrix(df, title, figsize=(10, 3), ytick_values=None, ytick_labels=None):
    plt.figure(figsize=figsize)
    plt.imshow(df.isnull().astype('int8').T.to_numpy(), aspect='auto', interpolation='nearest', cmap='viridis')
    if ytick_values is not None and ytick_labels is not None:
        plt.yticks(ytick_values, ytick_labels)
    else:
        plt.yticks([])
    plt.title(title)
    plt.xlabel('row index')
    plt.ylabel('column index')
    plt.show()


def cleanup(*var_names):
    for name in var_names:
        globals().pop(name, None)
    gc.collect()


# DATA2
주제 2. 구독서비스 프로덕트 데이터 분석

In [ ]:
# DATA_DIR is defined in the helper cell above.

parquet_file_sizes = get_parquet_file_sizes(DATA_DIR)
sorted_dict = dict(sorted(parquet_file_sizes.items(), key=lambda item: item[1], reverse=True))
for filename, size in sorted_dict.items():
    print(f'{filename:<50}: {size:.2f} MB')

| #  | 컬럼명              | 컬럼 설명                                                     | 컬럼 타입  | 비고                                                                                                                                   |
|:--:|:-------------------|:-------------------------------------------------------------|:---------|:---------------------------------------------------------------------------------------------------------------------------------------|
| 1  | **city**           | 접속 위치                                                    | Object   | 국내 기준 “시” 단위로 제공하나, 서울의 경우 “구”가 특정되는 경우는 “구” 단위로 표기                                                |
| 2  | **client_event_time** | 타임스탬프 (UTC+0 기준)                                   | Object   | -                                                                                                                                     |
| 3  | **country**        | 접속 국가                                                    | Object   | -                                                                                                                                     |
| 4  | **device_carrier** | 사용 이동통신사                                             | float64  | -                                                                                                                                     |
| 5  | **device_family**  | 사용 기기 유형                                              | Object   | -                                                                                                                                     |
| 6  | **device_type**    | 사용 기기                                                   | Object   | -                                                                                                                                     |
| 7  | **event_type**     | 이벤트 유형                                                 | Object   | 테이블명과 동일                                                                                                                       |
| 8  | **language**       | 사용자 설정 언어                                            | Object   | -                                                                                                                                     |
| 9  | **os_name**        | 사용 브라우저 이름                                          | Object   | -                                                                                                                                     |
| 10 | **os_version**     | 사용 브라우저 버전                                          | Object   | -                                                                                                                                     |
| 11 | **platform**       | 사용 플랫폼                                                | Object   | -                                                                                                                                     |
| 12 | **user_id**        | 유저 id                                                    | Object   | -                                                                                                                                     |

앰플리튜드 로그, GA로그와 유사한 형태

In [ ]:
def load_preprocessing(file_name):
    df = read_topic2_parquet(file_name)

    display(df.head(3))

    df['client_event_time'] = pd.to_datetime(df['client_event_time']) + datetime.timedelta(hours=9)

    print(f"\n\n시간최소 : {df['client_event_time'].min()} , 시간최대 :{df['client_event_time'].max()}\n")

    isn = df.isnull().sum()
    isn_ratio = (df.isnull().sum() / df.shape[0] * 100).round(1).astype('int').astype('str') + '%'
    unq = df.nunique()
    top_frq = object_top_frequency(df)

    pre_df = pd.concat([isn, isn_ratio, unq, top_frq], axis=1)
    pre_df.columns = ['결측치', '결측치(%)', '유니크값', '최빈값']
    pre_df.index.name = 'shape : ' + str(df.shape)
    display(pre_df)
    return df


# enter.main_page.csv
- 서비스 메인페이지 진입

In [ ]:
emp = load_preprocessing('enter_main_page.csv')

- 최소시간이 1978년도?
- 모두 웹 / 이벤트 타입 동일
- 이동통신사 결측치 100%
- 유저id 결측치가 75%이다. 이는 로그인 정보가 없는 상태인 유저들이라고 판단해야할 듯 / 비로그인상태
- 국적이 172개나 된다..?

In [ ]:
emp['client_event_time'].dt.year.value_counts()

In [ ]:
cleanup('emp')


1978년도는 데이터 이슈라고 보는게 맞을듯

# enter.signup_page.csv
- 회원가입 페이지 진입

In [ ]:
esp = load_preprocessing('enter_signup_page.csv')

- 이동 통신사는 결측치항상 0 인듯
- user_id 모두 nan이다. 회원가입 후 user_id 부여되는 듯
- 도시와 국가 최빈값을 보니 국내 위주의 서비스로 추측
- 회원가입페이지 진입 후 회원가입 완료 유저에 대해 퍼널 연결이 가능할지? -> complete.sighup과 시간대별 비교하는 수 밖에 없다. 데이터 조인은 불가     

In [ ]:
esp['client_event_time'].dt.date.value_counts().sort_index().plot(figsize=(15,5))
plt.title('기간별 회원가입 진입 시점')
plt.ylabel('counts')
plt.show()

- 주말과 주중의 주기가 존재하는듯?
- 피크치는 기간에는 특정 프로모션을 했던것이 아닌가 추측 해볼 수 있음

In [ ]:
esp['client_event_time'].dt.weekday.value_counts().sort_index().plot(kind='bar',figsize=(15,5)) # Monday=0, Sunday=6
plt.ylim(60000)
plt.title('기간별 회원가입 진입 시점')
plt.ylabel('counts')
plt.show()

토요일이 가장 낮은 가입진입
- 국가별로 시차 확인
- web 서비스이기 때문에 모바일 기종으로 가입하는 경우는 낮을 것이라 추측 -> 확인필요
- 데스크탑 vs 모바일의 다양한 기준으로 특성 차이 비교해보기

# complete.signup.csv
- 회원가입 완료

| 테이블명                                                | 테이블 설명                        | 컬럼명             | 타입     | 컬럼 설명                | 비고                                         |
|:--------------------------------------------------------|:-----------------------------------|:-------------------|:---------|:-------------------------|:---------------------------------------------|
|  **complete.signup**                                     | 회원가입 완료                      | type              | object   | 로그인 방식             | -                                           |

In [ ]:
csu = load_preprocessing('complete_signup.csv')

68%의 결측치가 일정하게 존재하는 경우들이 있다. 이는 시스템상의 업데이트로 수집 시작한 기간이 달라 발생했을 것이라 유추

In [ ]:
# missingness plotting is handled by plot_missing_matrix in the helper cell.


In [ ]:
plot_missing_matrix(
    csu.sort_values('client_event_time').reset_index(drop=True),
    'complete.signup.csv 컬럼별 결측치',
)


- 특정기간대에 결측치가 발생하는 것은 아님
- 결측치가 있는 행은 대체로 같은 컬럼들에서 켤측치를 가진다

In [ ]:
csu['type'].value_counts()

가입 경로를 확인해봤을 때, 카카오,네이버가 많다 -> 대한민국 대상 서비스
- test의 경우 개발단계의 데이터가 필터 안됐던것으로 보임 (데이터 qa팀 화이팅)
- 회원가입 시점과 비교해보기
- 국가정보도 확ㅇ니해보기~

In [ ]:
esp['client_event_time'].dt.date.value_counts().sort_index().plot(figsize=(15,5),label='회원가입페이지 유입')
csu['client_event_time'].dt.date.value_counts().sort_index().plot(figsize=(15,5),label='회원가입 완료')
plt.legend()
plt.title('회원가입페이지 유입 및 완료')
plt.ylabel('counts')
plt.show()

추세는 얼추 비슷해 보인다. 실질적인 비율 상의 차이가 있는지 확인해볼 필요가 있다.
- 유입인원 대비 가입인원을 하나의 지표로 두고 기간별 차이가 있는지 확인해볼만함
- 유입인원은 튀는데 (eg. 23년 11월쯤?) 가입완료는 적은 경우

In [ ]:
csu.user_id.value_counts()

In [ ]:
cleanup('esp', 'csu')


모든 User_id는 고유값을 가진다 -> 하나의 id는 1개의 회원 id값을 가짐

# enter.content_page
- 콘텐츠 개별 페이지 진입

| 테이블명 (이벤트명)                                | 테이블 설명                            | 컬럼명               | 타입    | 컬럼 설명                                         | 비고      |
|:----------------------------------------------------|:----------------------------------------|:--------------------|:--------|:----------------------------------|:----------|
| **enter.content_page**                              | 콘텐츠 개별 페이지 진입                 | content.id         | object | 콘텐츠 id                                       | -    |

In [ ]:
ecp = load_preprocessing('enter_content_page.csv')

- 유저 결측치가 25% 존재한다. 이를 어떻게 해석해야할지? 회원가입안한 유저가 컨텐츠에 속하게된건가
- 전체행은 191만행이다. 이중 유니크 유저는 7.2만으로 한 유저가 여러 컨텐츠에 포함이 되어 있을 것이다
- 전체 컨텐츠는 208개이다.
    - 컨텐츠는 처음부터 모두 존재했을지? 아니면 특정시기부터 시작되었을지 확인할 필요가 있다.
- 인기있는 컨텐츠는 무엇인지 확인해보기

In [ ]:
ecp.user_id.value_counts().head(10)

상위 2개 유저는 다른 유저 대비해서 상당히 많은 컨텐츠 접근이 있었다.
- 헤비유저 or 개발진 or 내부 공용아이디일 수 있다

In [ ]:
# 최고 컨텐츠 빈도 유저 데이터 확인
ecp[ecp.user_id =='4f74b6e2f650f4c8db87579e44f5b7d6'].sort_values('client_event_time').head(30)[['client_event_time','content.id','user_id']].head(20)

동일한 컨텐츠에 대해 10분간격으로 로그가 계속 남는다     
- 동일한 컨텐츠에 계속 유입되는 이유가 무엇인지 확인 불가 (아마 개발자 디버그?)
- 이상치로 봐도 될듯
- 동일한 컨텐츠에 지속적으로 로그를 남기는 시스템인지도 확인이 필요하다

In [ ]:
ecp.user_id.dropna().value_counts().describe()

유저별로 콘텐츠 접속에 대한 분포는 중위수가 3건, 평균 19건이다. max 2.9만건은 특이케이스 같음 자세히 볼필요있음

In [ ]:
#상위 7개의 컨텐츠가 7만명 중 1만명 이상이 시작했다.
ecp[['user_id','content.id']].dropna().drop_duplicates()['content.id'].value_counts().head(10)

In [ ]:
cleanup('ecp')


# click.content_page_start_content_button
- 콘텐츠 시청하기 버튼 클릭

| 테이블명 (이벤트명)                                | 테이블 설명                            | 컬럼명               | 타입    | 컬럼 설명                                         | 비고      |
|:----------------------------------------------------|:----------------------------------------|:--------------------|:--------|:----------------------------------|:----------|
| **click.content_page_start_content_button**         | 콘텐츠 시청하기 버튼 클릭               | content.id         | object | 콘텐츠 id                                       | -     |
| **click.content_page_start_content_button**         | 콘텐츠 시청하기 버튼 클릭               | button_name        | object | 버튼 이름                                        | 콘텐츠 페이지 개편 전   |
| **click.content_page_start_content_button**         | 콘텐츠 시청하기 버튼 클릭               | button.name        | object | 버튼 이름                                        | 콘텐츠 페이지 개편 후  |

In [ ]:
csb = load_preprocessing('click_content_page_start_content_button.csv')

- 컨텐츠 시청하기 버튼의 경우 유니크 값이 165개이다. 컨텐츠 개별페이지 진입시 유니크값은 208개였는데 일부만 버튼이 있나? 싶음
- 유저id의 결측치는 25%이다.

In [ ]:
plot_missing_matrix(
    sort_df,
    '콘텐츠 시청하기 버튼 클릭 데이터의 결측치',
    ytick_values=[0, 30000, 80000, 170000],
    ytick_labels=list(csb.sort_values('client_event_time')['client_event_time'].dt.date.iloc[[0, 30000, 80000, 170000]].values),
)


button.name의 경우 개편 후       
button_name의 경우 개편 전의 데이터를 나타낸다.    

대략 22년 5월부터 22년 12월까지는 관련 정보가 저장되어 있지 않다 -> 로그 기록이 없었던 듯

In [ ]:
csb['button.name'].unique()

In [ ]:
csb.button_name.unique()

In [ ]:
cleanup('sort_df', 'csb')


버튼은 각유저의 활성상태에 따른 이름 차이가 존재한다.

# click.content_page_more_review_button
- 콘텐츠 후기 더보기 버튼 클릭

| 테이블명 (이벤트명)                                | 테이블 설명                            | 컬럼명               | 타입    | 컬럼 설명                                         | 비고      |
|:----------------------------------------------------|:----------------------------------------|:--------------------|:--------|:----------------------------------|:----------|
| **click.content_page_start_content_button**         | 콘텐츠 시청하기 버튼 클릭               | content.id         | object | 콘텐츠 id                                       | -     |
| **click.content_page_start_content_button**         | 콘텐츠 시청하기 버튼 클릭               | button_name        | object | 버튼 이름                                        | 콘텐츠 페이지 개편 전   |
| **click.content_page_start_content_button**         | 콘텐츠 시청하기 버튼 클릭               | button.name        | object | 버튼 이름                                        | 콘텐츠 페이지 개편 후  |

In [ ]:
cpmr = load_preprocessing('click_content_page_more_review_button.csv')

- 유져 아이디가 88% 결측치
    - 로그인 하지 않은 상태일 때 후기를 많이 살펴본다는 의미?
    - Chrome Headless는 크롤러로 볼 수 있다.
    - 후기 수집 크롤러 ㅜㅅㅜ 나빠요

In [ ]:
cpmr[cpmr.os_name =='Chrome Headless']['client_event_time'].dt.date.value_counts().sort_index()

In [ ]:
cleanup('cpmr')


3일동안 크롤러의 데이터가 로그의 50%정도의 비율로 존재

# enter.payment_page

결제 페이지 진입


| 테이블명 (이벤트명)                                | 테이블 설명                            | 컬럼명               | 타입    | 컬럼 설명                                         | 비고      |
|:----------------------------------------------------|:----------------------------------------|:--------------------|:--------|:----------------------------------|:----------|
| **enter.payment_page**                              | 결제 페이지 진입                        | -                  | -      | -                                                | -        |

In [ ]:
epp = load_preprocessing('enter_payment_page.csv')

- 관련 로그가 22년 11월 부터 수집 / 다른 대부분 데이터는 21년도부터 있어서 연결해서 살펴보기 어려운 부분이 있음, 데이터들의 공통된 시간대를 살펴보는것 필요할듯
- 결제 완료 데이터와 묶어서 결제 페이지 이탈율을 확인 해볼 수 있음
- 같은 유저가 반복적으로 결제페이지를 들어갔다면 결제 오류 or 결제 갈등(?) or 개발자..

In [ ]:
epp['user_id'].value_counts()

In [ ]:
# max 유저
# 일단위로 결제페이지를 들락날락 -> QA인듯
epp[epp.user_id =='233094e3798cb9f681d9d1466e276ed8'].sort_values('client_event_time').head(30)

In [ ]:
epp['client_event_time'].dt.date.value_counts().sort_index().plot(figsize=(10,5))
plt.title('결제 페이지 진입 로그 일별 횟수')
plt.show()

In [ ]:
cleanup('epp')


23년 2,3분기에 상대적으로 떨어짐 / 다른기간이 프로모션등이 있었는지 확인필요

# complete.subscription
- 첫 결제 완료
- 요금제 종류, 쿠폰 종류, 쿠폰 혜택 종류, 정상가, 할인가, 할인액, 결제수단 타입
- 첫 결제가 중복인 경우는 첫 결제 환불 후 다시 결제한 케이스로 간주


| 테이블명 (이벤트명)                                | 테이블 설명                            | 컬럼명               | 타입    | 컬럼 설명                                         | 비고      |
|:----------------------------------------------------|:----------------------------------------|:--------------------|:--------|:----------------------------------|:----------|
| **complete.subscription**                         | 첫 결제 완료             | plan.price            | int64 | 정상가                                          | -|
| **complete.subscription**                         | 첫 결제 완료             | paid_amount           | int64 | 실제 결제 금액                                  | -|
| **complete.subscription**                         | 첫 결제 완료             | coupon.discount_amount | int64| 할인액                                          | 정상가 - 실제 결제 금액|
| **complete.subscription**                         | 첫 결제 완료             | pg.type               | object| 결제수단 타입                                   | A, B, C는 각기 다른 간편결제 수단|

In [ ]:
cs = load_preprocessing('complete_subscription.csv')

- 22년 1월부터 23년 12월말까지 첫결제 데이터

In [ ]:
(cs['plan.price'] - cs['paid_amount'] != cs['coupon.discount_amount']).sum()

할인금액과 실제 결제 금액간의 차이가 존재하는 행은 없다

첫 결제가 중복인 경우는 첫 결제 환불 후 다시 결제한 케이스 확인해보기

In [ ]:
multi_pay = cs.user_id.value_counts()[cs.user_id.value_counts() != 1].to_frame().rename(columns ={'count':'첫 결제 완료 횟수'})
multi_pay

In [ ]:
print(f'첫 결제를 취소하고 여러번 한 비율은 {round(multi_pay.shape[0] / 13881,2)* 100} % 이다')

In [ ]:
# ccea16701cf72a9027d7d5f2c3d3019c 유저의 첫결제 완료 시점 살펴보기
cs[cs.user_id =='ccea16701cf72a9027d7d5f2c3d3019c'][['client_event_time']].sort_values('client_event_time')

첫결제를 다수 진행한 유저는 첫 결제 시점이 넓은 기간에 걸쳐 존재한다.    
이러한 경우 구매 리텐션을 확인하는데 첫 구매시점을 어떤 시점으로 봐야하는지 어려움이 있을 수 있다.     
이런 유저를 제외하고 살펴볼지 포함하고 살펴볼지는 추가적인 분석작업을 통해 진행 해보길

# renew.subscription.csv

- 정기 결제 완료

| 테이블명 (이벤트명)                               | 테이블 설명              | 컬럼명                 | 타입   | 컬럼 설명                                       | 비고 |
|:--------------------------------------------------|:------------------------|:-----------------------|:------|:------------------------------------------------|:--|
| **renew.subscription**                            | 정기 결제 완료           | plan.price            | int64 | 정상가                                          | -|
| **renew.subscription**                            | 정기 결제 완료           | paid_amount           | int64 | 실제 결제 금액                                  | -|
| **renew.subscription**                            | 정기 결제 완료           | coupon.discount_amount | int64| 할인액                                          | 정상가 - 실제 결제 금액|                              
| **renew.subscription**                            | 정기 결제 완료           | pg.type               | object| 결제수단 타입                                   | A, B, C는 각기 다른 간편결제 수단|

In [ ]:
rs = load_preprocessing('renew_subscription.csv')

- pg.type은 결측치가 많아서 활용에 어려움이 있어보임
- 모든 정기 결제 고객은 첫구매 데이터가 존재하는지 확인필요
- 기간은 22년 9월부터 23년 12월말까지

In [ ]:
print(f'재구매 리스트에는 있는데 첫 구매 리스트에는 없는 유저 숫자는 {len(set(rs.user_id.unique()) - set(cs.user_id.unique()))}명이다. 이는 재구매 인원의 {round(len(set(rs.user_id.unique()) - set(cs.user_id.unique())) / rs.user_id.nunique()* 100,2)} %이다.' )

첫구매가 존재하는 재구매 인원만 필터하여 재구매 하는 유저의 특성을 그렇지 않은 유저들과 비교해서 살펴보면 좋을 듯
- 어떤 유저 특성을 가지는지
- 어떤 컨텐츠를 많이 소비 했는지
- 액티브 정도는 얼마나 되는지

# resubscribe.subscription
만료 후 재구독 완료

In [ ]:
rss = load_preprocessing('resubscribe_subscription.csv')

- 구독 시작이 7498명인데, 재구독이 761명 뿐이다.
- 12개월 구독권도 있기에 아직 결제 시점이 도달하지 않은 유저가 존재할 수 있음

In [ ]:
print(f'재구독은 했지만 첫구독은 정보가 없는 유저는 {len(set(rss.user_id) - set(rs.user_id))}명이다.\n해당유저들이 free_trial로 시작했는지 확인 필요')

In [ ]:
cleanup('cs', 'rs', 'rss')


[링크 텍스트](https://)# start.free_trial
- 서비스 무료체험 시작

| 테이블명 (이벤트명)                               | 테이블 설명              | 컬럼명                 | 타입   | 컬럼 설명                                       | 비고 |
|:--------------------------------------------------|:------------------------|:-----------------------|:------|:------------------------------------------------|:--|
| **start.free_trial**                              | 서비스 무료체험 시작      | plan.price            | int64 | 정상가                                          | -|
| **start.free_trial**                              | 서비스 무료체험 시작      | plan.type             | object| 요금제 종류                                    | -|
| **start.free_trial**                              | 서비스 무료체험 시작      | trial.type           | object| 무료체험 방식                                   | "- type A(구버전): 특정 컨텐츠 하나 영구 무료체험<br>- type B(신버전): 기간 단위로 전체 컨텐츠 경험(중복X)"|


In [ ]:
sf = load_preprocessing('start_free_trial.csv')

- 무료 체험 이후에 정기 구독을 시작하는 경우는 있는가? 있다면 그 비율은?
- 정기 구독이후에 무료체험을 하는 유저는 있는가? (데이터 오류로 보임 체크 필요)
- 무료 체험을 하는 유저는 어떤 컨텐츠를 많이 소비하는가?

In [ ]:
pd.crosstab(sf['plan.price'],sf['plan.type'])

In [ ]:
cleanup('sf')


유저의 행동별 데이터 확인 필요    
1. 무료이용 -> 첫구독 -> 재구독
2. 무료이용 -> 재구독
3. 첫구독 -> 재구독
4. 첫구독

각 항목별 예외 케이스 데이터가 분명 존재할 듯 잘 나눠서 확인 필요

# start.content
- 콘텐츠 수강 시작

| 테이블명 (이벤트명)                               | 테이블 설명              | 컬럼명                 | 타입   | 컬럼 설명                                       | 비고 |
|:--------------------------------------------------|:------------------------|:-----------------------|:------|:------------------------------------------------|:--|
| **start.content**                                 | 콘텐츠 수강 시작         | content.id            | object| 콘텐츠 id                                       | -|
| **start.content**                                 | 콘텐츠 수강 시작         | content.difficulty    | int64 | 콘텐츠 난이도                                   | -|

In [ ]:
sc = load_preprocessing('start_content.csv')

- 유저 유니크 숫자는 4.2만이다
- 유저별 content.id는 시작하는 경우가 유일할 것으로 생각되는데 실제 그런지 확인 필요

In [ ]:
result = sc[['content.difficulty','user_id']].drop_duplicates()['content.difficulty'].value_counts().to_frame()
result.index.name ='난이도에 따른 시작유저 숫자'
result

In [ ]:
sc[['user_id','content.id']].drop_duplicates()['user_id'].value_counts().head(10) # 유저의 컨텐츠 시작횟수

In [ ]:
sc[['user_id','content.id']].value_counts() # 한 유저가 동일한 컨텐츠를 여러번 시청한 경우가 있다.

In [ ]:
# 위 케이스1
sc[(sc.user_id =='23d69a936e8267e6f2ec7037bd3b54ca') & (sc['content.id'] == '4641438a6c81ef572d997dbdc9100f8b')]

1초 내에 여러번 응답이 된 것으로 보인다. 로그 이슈로 보임

In [ ]:
# 위 케이스2
# 일본..? 뭐고.. 12월 24일에 여러회차 시도했던것으로 보임
sc[(sc.user_id =='5c2ce4062550e53fa30c3f53b8c9b181') & (sc['content.id'] == '61b6463287573f00de13a930805a52d6')]

In [ ]:
# 위 케이스3
# 일본..? 뭐고.. 12월 24일에 여러회차 시도했던것으로 보임
sc[(sc.user_id =='40651151784699b184c58ddcd2dedf9d') & (sc['content.id'] == 'c269eb6df3a374b464f7c18f12fa398f')]

In [ ]:
cleanup('sc')


1초 내에 여러번 응답이 된 것으로 보인다. 로그 이슈로 보임
- 대체로 로그 이슈일 것이다. 중복 제거하면 유니크 케이스에 대해 확인 가능

추가질문
- 각 유저는 어떤 정기권을 통해 어떤 컨텐츠로 유입이 되었는가?
- 인기 많은 처음 시작 컨텐츠는 무엇인가
- 컨텐츠 난이도에 따른 컨텐츠 시작 양상 특징은?
- 여러 컨텐츠를 들은 유저들은 수강 패턴이 존재하는지?

# enter.lesson_page
- 콘텐츠는 여러개의 레슨로 구성되어있습니다.

| 테이블명 (이벤트명)                               | 테이블 설명              | 컬럼명                 | 타입   | 컬럼 설명                                       | 비고 |
|:--------------------------------------------------|:------------------------|:-----------------------|:------|:------------------------------------------------|:--|
| **enter.lesson_page**                             | 레슨 시작                | content.id            | object| 콘텐츠 id                                       | -|
| **enter.lesson_page**                             | 레슨 시작                | lesson.id             | object| 레슨 id                                         | -|
| **enter.lesson_page**                             | 레슨 시작                | is_free_trial         | boolean| 무료 공개 레슨 진입 여부                        | 특정 기준 시점으로 사용 안 함|
| **enter.lesson_page**                             | 레슨 시작                | is_trial              | boolean| “무료 공개 레슨 + 미구독 유저”인지 여부         | -|

In [ ]:
el = load_preprocessing('enter_lesson_page.csv')

- 행이 2천만개 있다.
- user_id가 없는 데이터가 7%... / 회원 가입을 하지 않은 유저인 것인지 로그 누락인지는 별도 확인필요
- 181개의 content와 5479개의 lesson가 존재
    - 하나의 레슨은 하나의 컨텐츠에만 종속되는지 확인 필요


In [ ]:
el[['user_id','content.id','lesson.id']].dropna().value_counts().head(20)

하나의 유저가 한개의 컨텐츠의 하나의 레슨을 8318번 시작했다? 하루에 20번씩해도 2년이다. 뭐고 이거

In [ ]:
# 최상위 데이터의 샘플 필터
el[(el.user_id =='4f74b6e2f650f4c8db87579e44f5b7d6') & (el['content.id'] =='595cb4bbfc83e683b0314ca1312cfbde') & (el['lesson.id'] =='bef2ec4ced488c2bd28cbba9752f73c2')].sort_values('client_event_time').head(10)\
        [['city','client_event_time','user_id','lesson.id']]

해당 유저는 다양한 레슨에 대해서 다양하게 시도를 했다. 용인시에 사는 서비스 개발자로 추정 (디버깅 or QA 과정의 데이터가 섞인 것으로 보임 해당 데이터들 필터 해줘야함)

In [ ]:
counts =el[el.user_id =='4f74b6e2f650f4c8db87579e44f5b7d6'].shape[0]
print(f'이상치 유저 한명이 발생시킨 시작로그 개수는 {counts}개이다. 이런 데이터는 걸러줘야한다')

In [ ]:
# 두번째로 로그 많은 유저
el[el.user_id =='1400ef9237d90f6e326678e7758e76ce']['lesson.id'].value_counts()

In [ ]:
cleanup('el')


# complete.lesson
- 레슨 완료	콘텐츠

| 테이블명 (이벤트명)                               | 테이블 설명              | 컬럼명                 | 타입   | 컬럼 설명                                       | 비고 |
|:--------------------------------------------------|:------------------------|:-----------------------|:------|:------------------------------------------------|:--|
| **complete.lesson**                               | 레슨 완료                | content.id            | object| 콘텐츠 id                                       | -|
| **complete.lesson**                               | 레슨 완료                | lesson.id             | object| 레슨 id                                         | -|

In [ ]:
cl = load_preprocessing('complete_lesson.csv')

- 레슨 완료 데이터는 540만행
- 레슨 유니크 값은 5050개다. 레슨 시작 데이터에서 레슨 유니크 값은 5479개이다.
    - 10%정도의 데이터는 레슨 완료 케이스가 하나도 없다? -> 실제 그런것인지 확인필요
        - 레슨 완료 케이스가 없는 레슨 수강하는 동안 다른 레슨을 완료한 유저가 존재한다면 주장이 옳다고 볼 수 있다.

# click.lesson_page_related_question_box
- 레슨 페이지 내 질문 목록 클릭

| 테이블명 (이벤트명)                               | 테이블 설명              | 컬럼명                 | 타입   | 컬럼 설명                                       | 비고 |
|:--------------------------------------------------|:------------------------|:-----------------------|:------|:------------------------------------------------|:--|
| **click.lesson_page_related_question_box**        | 레슨 페이지 내 질문목록 클릭 | content.id        | object| 콘텐츠 id                                       | -|
| **click.lesson_page_related_question_box**        | 레슨 페이지 내 질문목록 클릭 | lesson.id         | object| 레슨 id                                         | -|
| **click.lesson_page_related_question_box**        | 레슨 페이지 내 질문목록 클릭 | question.id       | object| 질문 id                                         | -|

In [ ]:
clpr = load_preprocessing('click_lesson_page_related_question_box.csv')

In [ ]:
cleanup('cl', 'clpr')


- 질문 목록 클릭의 활성화 정도?
- 레슨 완료율과의 연관성이 존재?

# end.content
- 콘텐츠 수강 완료
- 모든 레슨를 시청하면 해당 테이블에 로그가 쌓임

| 테이블명 (이벤트명)                               | 테이블 설명              | 컬럼명                 | 타입   | 컬럼 설명                                       | 비고 |
|:--------------------------------------------------|:------------------------|:-----------------------|:------|:------------------------------------------------|:--|
| **end.content**                                   | 수강 완료                | content.id            |       | 콘텐츠 id                                       | -|

In [ ]:
ec = load_preprocessing('end_content.csv')

- 시작한 컨텐츠 id가 151개인데, 종료된 컨텐츠 id가 160개 이다. ㅎㅎ ;;
- 컨텐츠와 레슨의 완료의 정확한 기준(로그가 발생하는 트리거)을 알아야 추가 분석이 가능할 거 같다. 현실적으로 확인이 어렵기에 추정을 통해 진행하자
- start와 비교해서 이탈하는 유저는 어느 정도 될지?
- 중도 이탈 유저의 특징

# click.cancel_plan_button
- 구독 취소 버튼 클릭
- 구독 취소 버튼을 클릭했다면 구독 취소가 이루어진 것으로 간주

| 테이블명 (이벤트명)                               | 테이블 설명              | 컬럼명                 | 타입   | 컬럼 설명                                       | 비고 |
|:--------------------------------------------------|:------------------------|:-----------------------|:------|:------------------------------------------------|:--|
| **click.cancel_plan_button**                      | 구독 취소 버튼 클릭       | -                     | -     | -                                              | -|

In [ ]:
ccp = load_preprocessing('click_cancel_plan_button.csv')

In [ ]:
cleanup('ec', 'ccp')


- 14643명이 구독을 취소했다.14643명이나 구독을 했었나..?
- 첫구독 7498 / 재구독 761 / 무료시작 16309
- 무료시작한 유저도 구독 정지를 한 케이스가 있나보다.
- 구독종료를 한 이후에 다시 구독을 시작 한 경우도 있는지 체크해보자

## DATA2 참고
- 전체 데이터에 포함된 user_id unique 파악
- 전체 데이터의 공통된 기간 영역
- 존재하는 기간이 다른 데이터 간에 해석 가능한 방법이 있을지 체크
- 개발, QA로 보이는 user_id들에 대해 판단하고 제거해주는 과정 필요
- 웹 크롤러로 보이는 (chrome headless) 로그들을 제거할 필요 있음
- 해외 이용의 비중이 적다면 확실히 제거하고 국내, 해외를 최상단에서 구분해서 분석하면 좋을듯!